# Analyzing Flood Impact on Infrastructure

In [1]:
# Import libraries
import ee
import geemap
import geemap.chart as chart
import pandas as pd
import geopandas as gpd

import os

## . Connect to GEE

In [2]:
# Authenticate GEE
ee.Authenticate()

#Initialize GEE
ee.Initialize()

## . Flood Parameters

In [3]:
# Pre-flood date
before_start = "2022-08-01"
before_end   = "2022-09-10"

# Flood date
after_start = "2022-09-14"
after_end   = "2022-11-18"

# Cloud percentage value
cloud_perc = 58

# Radius for speckle filtering (Sentinel-1)
speckle_radius = 50 # Unit is meter



## . Visualization Parameters

In [4]:
# Boundary (AOI)
vis_params_aoi = {'color': 'red', 'fillColor': '00000000'}

# Sentinel-2
vis_params_s2_rgb = {"min": 0.1, "max": 0.6, "bands": ["B4", "B3", "B2"], "gamma": 0.9} # True Colour - RGB
vis_params_s2_fcc = {"min": 0.01, "max": 0.6, "bands": ["B8", "B4", "B3"], "gamma": 0.9} # False colour

# Sentinel-1
vis_params_s1_vv = {"min": -20, "max": 5, "bands": ["VV"], "palette": ["black", "white"]}
vis_params_s1_vh = {"min": -20, "max": 5, "bands": ["VH"], "palette": ["black", "white"]}

# Landsat-8
vis_params_l8_rgb = {"bands": ["SR_B4", "SR_B3", "SR_B2"], "min": 0, "max": 0.3}
vis_params_l8_fcc = {"bands": ["SR_B5", "SR_B4", "SR_B3"], "min": 0, "max": 0.3}

# GSW Permanent Water
vis_params_perm_water = {'min': 0,'max': 1,'palette': ["cyan"]}

# Flood mask
vis_params_flood = {"min": 0, "max": 1, "palette": ["white", "blue"]}

# Population Density
vis_params_pop = {"min": 0, "max": 100, "palette": ["white", "yellow", "orange", "red", "purple"]}

# DEM
vis_params_srtm_dem = {"min": 0, "max": 3000, "palette": ["blue", "green", "yellow", "brown"]}
vis_params_cop_dem = {"min": 0, "max": 3000, "palette": ["white", "yellow", "orange", "red"]}

# ESA WorldCover 2021
vis_params_landcover = {
    "min": 10,
    "max": 100,
    "palette": [
        "#006400",  # 10: Tree cover
        "#ffbb22",  # 20: Shrubland
        "#ffff4c",  # 30: Grassland
        "#f096ff",  # 40: Cropland
        "#fa0000",  # 50: Built-up
        "#b4b4b4",  # 60: Bare / sparse vegetation
        "#f0f0f0",  # 70: Snow and ice
        "#0064c8",  # 80: Permanent water bodies
        "#0096a0",  # 90: Herbaceous wetland
        "#00cf75",  # 95: Mangroves
        "#fae6a0"   # 100: Moss and lichen
    ]
}

## . Visualization Functions

In [5]:
# Function to create a base map
def create_basemap():
    map = geemap.Map(center=(7.8175, 6.7730), zoom=10)
    map.add_basemap("SATELLITE")

    return map


# Function to display a raster layer




# Function to display a vector layer




In [6]:
# Function to display an image histogram


## . Flood Functions 

## . Area of Interest (AOI)

In [7]:
# AOI (GeoJSON)
aoi_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [6.853409, 7.676968],
            [6.685181, 7.676968],
            [6.685181, 7.890588],
            [6.853409, 7.890588],
            [6.853409, 7.676968]
        ]
    ]
}

# Convert to GEE Feature
aoi_feature = ee.Feature(aoi_geojson)


aoi_geometry = aoi_feature.geometry()

# Default location to show the map
map_center_coords = 7.8175, 6.7730


In [8]:
# Show AOI on the map
aoi_map = geemap.Map(center=(7.8175, 6.7730), zoom=10)
aoi_map.add_basemap("SATELLITE")

aoi_map.addLayer(aoi_feature, vis_params_aoi, "AOI Flood")
aoi_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

### . Get Admin Boundaries

In [9]:
# Get Nigeria Admin Boundary from FAO
fao_bdry_l0 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level0") # Adm0
fao_bdry_l1 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1") # Adm1
fao_bdry_l2 = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2") #Adm2

# Get the boundary of Nigeria from FAO GAUL
nga_bdry_l0 = fao_bdry_l0.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # Main Nigerian bdry
nga_bdry_l1 = fao_bdry_l1.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # States bdry
nga_bdry_l2 = fao_bdry_l2.filter(ee.Filter.eq("ADM0_NAME", "Nigeria")) # LGAs bdry

lokoja_by_name = nga_bdry_l2.filter(ee.Filter.eq("ADM2_NAME", "Lokoja"))

In [10]:

# Add Nigeria boundaries
aoi_map.addLayer(nga_bdry_l0, {"color": "black"}, "NGA Bdry L0")
aoi_map.addLayer(nga_bdry_l1, {"color": "blue"}, "NGA State")
aoi_map.addLayer(nga_bdry_l2, {"color": "green"}, "NGA LGA L2")

# Add Lokoja
aoi_map.addLayer(lokoja_by_name, {"color": "black"}, "Lokoja (Name)")

# Display map
aoi_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## . Satellite Image Download and Processing

### . Sentinel-2 Download and Pre-Process 

In [11]:
# Function to download Sentinel-2
def download_s2(start_date: str, end_date : str, cloud_perc: int, extent : ee.Feature | ee.Geometry):
    """
    Download Sentinel-2 images filtered by date, cloud cover, and geographic extent.

    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        cloud_perc (int): Maximum allowable cloud pixel percentage.
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.

    Returns:
        ee.ImageCollection: The filtered Sentinel-2 collection.
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                    .filterDate(start_date, end_date) # Dates filter
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_perc)) # Cloud filter
                    .filterBounds(extent)) # Boundary filter   

    img_count =  img_count = img_collection.size().getInfo() # Number of image in the collection

    print(f"There are {img_count} images in the Sentinel-2 image collection\n")
    
    return img_collection



# Function to apply cloud masking to Sentinel-2
# See reference: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_CLOUD_PROBABILITY
def mask_clouds_s2(image: ee.Image, cloud_thresh: int):
    """
    Apply cloud masking using cloud probability
    Uses the S2_CLOUD_PROBABILITY dataset to mask out cloudy pixels.

    Args:
        image (ee.Image): Sentinel-2 image with cloud probability band
        cloud_thresh (int): Cloud probability threshold (0-100)

    Returns:
        ee.Image: The cloud-masked image.
    """
    clear_threshold = cloud_thresh
    clouds = ee.Image(
            image.get("cloud_mask")
            ).select("probability")
    
    no_clouds = clouds.lt(clear_threshold)
    masked_image = image.updateMask(no_clouds).copyProperties(image, ["system:time_start"])
    
    return masked_image
    

# Masks from the 20m and 60m bands for bad data at scene edges
def mask_edges_s2(image : ee.Image):
    """
    Mask bad data at scene edges using 20m and 60m band masks.
    
    Args:
        image: ee.Image - Sentinel-2 image
    
    Returns:
        ee.Image - Image with edges masked
    """
    masked_image = image.updateMask(
                    image.select("B8A").mask().updateMask(image.select("B9").mask())
                    ).copyProperties(image, ["system:time_start"])
    
    return masked_image 


# Function to convert Sentinel-2 DN to reflectance
def scale_bands_s2(image: ee.Image):
        """
        Convert Sentinel-2 digital numbers to surface reflectance.
    
    Args:
        image: ee.Image - Sentinel-2 image with DN values
    
    Returns:
        ee.Image - Image with scaled reflectance values (0-1)
        """
        scaled_image = image.multiply(0.0001).copyProperties(image, ["system:time_start"]) # Multiply image by 0.0001 and copy "system:time_start" property 
        return scaled_image


# Function to resample bands in Sentinel-2 to 10m
def resample_bands_s2(image: ee.Image):
        """
        Resample 20m bands to 10m resolution using bilinear interpolation.
    
    Args:
        image: ee.Image - Sentinel-2 image with 10m and 20m bands
    
    Returns:
        ee.Image - Image with all bands at 10m resolution
        """
        bands_10m = image.select(["B2", "B3", "B4", "B8"]) # Select 10m bands
        bands_20m = image.select(["B5", "B6", "B7", "B8A", "B11", "B12"]) # Select 20m bands 
        resampled_image = bands_20m.resample("bilinear").reproject(
                                                            crs = bands_10m.projection(), 
                                                             scale=10)
        
        combined_10m = bands_10m.addBands(resampled_image) # Combine original 10m with 20m
        
        return combined_10m.copyProperties(image, ["system:time_start"])


# Function to compute spectral indices
def compute_indices_s2(image:ee.Image):
        """
        Compute spectral indices (NDVI, NDWI, MNDWI, EVI) for Sentinel-2 imagery.
    
    Args:
        image: ee.Image - Sentinel-2 image with required bands
    
    Returns:
        ee.Image - Original image with added spectral indices
        """
        
        ndvi = image.normalizedDifference(["B8", "B4"]).rename("ndvi")
        ndwi = image.normalizedDifference(["B3", "B8"]).rename("ndwi")
        mndwi = image.normalizedDifference(["B3", "B11"]).rename("mndwi")
        evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR":image.select("B8"),
            "RED": image.select("B4"),
            "BLUE": image.select("B2")
        }
        ).rename("evi")
        
        updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(image, ["system:time_start"])
        
        return updated_image 



In [12]:
# Download Sentinel-2 images
s2_img_col = download_s2(after_start, after_end, cloud_perc, aoi_geometry)

# Apply cloud masking to ALL IMAGES in the Sentinel-2 collection
# Cloud mask is typically applied using the cloud probability dataset
s2_img_col = s2_img_col.map(lambda img: mask_edges_s2(img))  # Edge masking

# Resample bands for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: resample_bands_s2(img))

# Rescale pixel values  for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: scale_bands_s2(img))
                            
# Compute Indices for ALL IMAGES in the Sentinel-2 collection
s2_img_col = s2_img_col.map(lambda img: compute_indices_s2(img))

There are 15 images in the Sentinel-2 image collection



In [13]:
# Convert Sentinel-2 images to a List
s2_img_list = s2_img_col.toList(s2_img_col.size())

In [14]:
# Create a median composite for visualization
s2_composite = s2_img_col.median().clip(aoi_feature)

# Visualize Sentinel-2 imagery

s2_map = geemap.Map(center= map_center_coords, zoom=11)
s2_map.add_basemap("SATELLITE")

s2_map.addLayer(s2_composite, vis_params_s2_rgb, "Sentinel-2 RGB (Post-flood)")
s2_map.addLayer(s2_composite, vis_params_s2_fcc, "Sentinel-2 False Color (Post-flood)")
s2_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s2_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

### . Sentinel-1 Download and Pre-process 

In [15]:
# Function to download Sentinel-1
def download_s1(start_date: str, end_date: str, extent: ee.Feature | ee.Geometry):
    """
    Download Sentinel-1 GRD images filtered by date, instrument mode, resolution, and geographic extent.
    
    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.
        
    Returns:
        ee.ImageCollection: The filtered Sentinel-1 collection.
    """
    img_collection = (ee.ImageCollection("COPERNICUS/S1_GRD")
                        .filterDate(start_date, end_date) # Dates filter
                        .filter(ee.Filter.eq("instrumentMode", "IW")) # InstrumentMode
                        .filterMetadata("resolution_meters", "equals", 10) # Resolution filter
                        .filterBounds(extent) # Boundary filter
                        .select(["VV", "VH"])) # Select polarizations

    # Number of images in the Sentinel-1 Collection
    img_count = img_collection.size().getInfo()
    
    print(f"There are {img_count} Sentinel-1 Images")

    return img_collection


# Function to apply Speckle filter to Sentinel-1 (focal median)
def focal_speckle_filter(image : ee.Image, radius : int,  unit: str = "meters"):
    """
    Focal Median filter with circle kernel.
    Apply focal median speckle filter to Sentinel-1 imagery using a circular kernel.
    
    Focal median filtering reduces speckle noise in SAR imagery by replacing
    each pixel with the median value of its neighboring pixels.
    
    Args:
        image (ee.Image): Sentinel-1 image to filter.
        radius (int): Radius of the circular kernel in specified units.
        unit (str): Units for the radius ('meters' or 'pixels'). Default is 'meters'.
        
    Returns:
        ee.Image: Filtered image with reduced speckle noise.
    """
    image_filtered = image.focal_median(radius, "circle", unit)
    image_filtered.copyProperties(image, image.propertyNames())

    return image_filtered

In [16]:
# Download Sentinel-1 imagery
s1_img_col = download_s1(after_start, after_end, aoi_geometry)

# Apply Speckle filter to ALL IMAGES in the Sentinel-1 collection
s1_img_col = s1_img_col.map(lambda img: focal_speckle_filter(img, speckle_radius, "meters"))


There are 4 Sentinel-1 Images


In [17]:
# Convert Sentinel-1 images to a List
s1_img_list = s1_img_col.toList(s1_img_col.size())

In [18]:
# Create a median composite for visualization
s1_composite = s1_img_col.median().clip(aoi_feature)

# Visualize Sentinel-1 imagery
s1_map = geemap.Map(center=map_center_coords, zoom=11)
s1_map.add_basemap("SATELLITE")

s1_map.addLayer(s1_composite, vis_params_s1_vv, "Sentinel-1 VV (Post-flood)")
s1_map.addLayer(s1_composite, vis_params_s1_vh, "Sentinel-1 VH (Post-flood)")
s1_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s1_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…


### . Landsat-8 Download Pre-Process

In [19]:
# Function to download Landsat-8
def download_l8(start_date: str, end_date: str, cloud_perc: int, extent: ee.Feature | ee.Geometry):
    """
    Download Landsat-8 Collection 2 Level 2 imagery filtered by date, cloud cover, and area.

    Args:
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        cloud_perc (int): Maximum allowed cloud cover percentage.
        extent (ee.Feature | ee.Geometry): Area of interest.

    Returns:
        ee.ImageCollection: Filtered Landsat-8 image collection.
    """
    img_collection = (
        ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUD_COVER", cloud_perc))
        .filterBounds(extent)
    )

    img_count = img_collection.size().getInfo()
    print(f"There are {img_count} images in the Landsat-8 image collection\n")

    return img_collection


# Band scaling factors for Landsat-8
def scale_bands_l8(image: ee.Image):
    """
    Apply scale factors to Landsat-8 optical and thermal bands.

    Args:
        image (ee.Image): Landsat-8 image.

    Returns:
        ee.Image: Scaled Landsat-8 image.
    """
    optical_bands = image.select(["SR_B.*"]).multiply(0.0000275).add(-0.2)
    thermal_bands = image.select(["ST_B.*"]).multiply(0.00341802).add(149.0)

    scaled_bands = image.addBands(optical_bands, None, True).addBands(thermal_bands, None, True)

    return scaled_bands.copyProperties(image, ["system:time_start"])


# Cloud Masking for Landsat-8
def mask_clouds_l8(image: ee.Image):
    """
    Mask clouds and cloud shadows in Landsat-8 SR imagery using the QA_PIXEL band.

    Args:
        image (ee.Image): Landsat-8 image.

    Returns:
        ee.Image: Cloud-masked image.
    """
    qa = image.select("QA_PIXEL")
    cloud = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    qa_mask = cloud.And(cloud_shadow)

    masked_image = image.updateMask(qa_mask).copyProperties(image, ["system:time_start"])

    return masked_image


# Function to compute spectral indices
def compute_indices_l8(image: ee.Image):
    """
    Compute spectral indices (NDVI, NDWI, MNDWI, EVI) for Landsat-8 imagery.

    Args:
        image (ee.Image): Landsat-8 image with required bands.

    Returns:
        ee.Image: Original image with added spectral index bands.
    """
    ndvi = image.normalizedDifference(["SR_B5", "SR_B4"]).rename("ndvi")
    ndwi = image.normalizedDifference(["SR_B3", "SR_B5"]).rename("ndwi")
    mndwi = image.normalizedDifference(["SR_B3", "SR_B6"]).rename("mndwi")

    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            "NIR": image.select("SR_B5"),
            "RED": image.select("SR_B4"),
            "BLUE": image.select("SR_B2")
        }
    ).rename("evi")

    updated_image = image.addBands([ndvi, ndwi, mndwi, evi]).copyProperties(
        image, ["system:time_start"]
    )

    return updated_image


In [20]:
# Download Landsat-8 images
l8_img_col = download_l8(after_start, after_end, cloud_perc, aoi_geometry)

# Apply cloud masking to ALL IMAGES in the Landsat-8 collection
l8_img_col = l8_img_col.map(lambda img: mask_clouds_l8(img))

# Rescale pixel values for ALL IMAGES in the Landsat-8 collection
l8_img_col = l8_img_col.map(lambda img: scale_bands_l8(img))

# Clip ALL IMAGES in the Landsat-8 collection to the AOI
l8_img_col = l8_img_col.map(lambda img: img.clip(aoi_geometry))

# Compute Indices for ALL IMAGES in the Landsat-8 collection
l8_img_col = l8_img_col.map(lambda img: compute_indices_l8(img))

There are 5 images in the Landsat-8 image collection



In [21]:
# Convert Landsat-8 images to a List 
l8_img_list = l8_img_col.toList(10)

# Create a median composite for visualization
l8_composite = l8_img_col.median().clip(aoi_geometry)

# Create map
l8_map = geemap.Map()
l8_map.centerObject(aoi_geometry, 11)

# Optional basemap
l8_map.add_basemap("SATELLITE")

# Add layers
l8_map.addLayer(l8_composite, vis_params_l8_rgb, "Landsat-8 RGB (Post-flood)")
l8_map.addLayer(l8_composite, vis_params_l8_fcc, "Landsat-8 False Color (Post-flood)")
l8_map.addLayer(aoi_geometry, {"color": "red"}, "AOI")

l8_map

Map(center=[7.783777212404713, 6.769295000001247], controls=(WidgetControl(options=['position', 'transparent_b…

## . Flood Mapping

### . Histogram Analysis

**Image Histogram for Sentinel-2**

In [22]:
# Image Histogram for Sentinel-2

# Extract the NDWI band from Sentinel 2 composite
ndwi = s2_composite.select('ndwi')

# Sample values from the NDWI layer
sampled_values_s2 = ndwi.sample(
    region=aoi_geometry, 
    scale=10, 
    numPixels=10000
)

# Options for the histogram plot
hist_options = {
    "title": "NDWI Distribution - Sentinel-2",
    "xlabel": "NDWI Value",
    "ylabel": "Pixel Count",
    "colors": ["#1d6b99"],
}

# Generate Histogram
hist_s2 = chart.feature_histogram(
    features=sampled_values_s2,          
    property="ndwi",         
    maxBuckets=50,                  
    **hist_options                    
)
print(hist_s2)

None


**Image Histogram for Sentinel-1**

In [23]:
# Image Histogram for Sentinel-1

# S1 VV band
s1_band_vv = s1_composite.bandNames().getInfo()[0] # VV band

# Sample values as features from VV band
sampled_values_s1 = s1_composite.sample(region=aoi_geometry, 
                            scale=10, 
                            numPixels=10000)

# Options for the histogram plot
hist_options = {
        "title": "Image Distribution",
        "xlabel": "Values",
        "ylabel": "Pixel Count",
        "colors": ["#1d6b99"],
    }

# Distribution (backscatter) values using histogram 


hist_s1 = chart.feature_histogram(
    features=sampled_values_s1,          
    property=s1_band_vv,         
    maxBuckets=50,                  
    **hist_options                    
)

print(hist_s1)


None


**Image Histogram for Landsat-8**

In [24]:
# Image Histogram for Landsat-8

# Extract NDWI band from Landsat-8 composite
ndwi_l8 = l8_composite.select('ndwi')

# Sample values from NDWI layer
sampled_values_l8 = ndwi_l8.sample(
    region=aoi_geometry,
    scale=30,          # IMPORTANT: Landsat-8 uses 30m
    numPixels=10000
)

# Options for the histogram plot
hist_options = {
    "title": "NDWI Distribution - Landsat-8",
    "xlabel": "NDWI Value",
    "ylabel": "Pixel Count",
    "colors": ["#1d6b99"],
}

# Generate Histogram
hist_l8 = chart.feature_histogram(
    features=sampled_values_l8,
    property="ndwi",
    maxBuckets=50,
    **hist_options
)

print(hist_l8)

None


### . Flood extraction using Thresholding Approach

**Sentinel-2**

In [25]:
# Flood extraction function for Sentinel-2

def extract_floodmask_s2(image: ee.Image, s2_ndwi_threshold: float, extent: ee.Feature | ee.Geometry):
    """
    Extract flood mask from Sentinel-2 imagery using NDWI thresholding approach.

    
    Args:
        image (ee.Image): Sentinel-2 image
        s2_ndwi_threshold (float): NDWI threshold for water detection (default 0.0). Higher values detect more water.
        extent (ee.Feature|ee.Geometry): The region to clip by

    Returns:
        ee.Image: Binary flood mask (1 = flood, 0 = non-flood).
    """
    # Create the mask where NDWI is greater than the threshold
    mask = image.select("ndwi").gt(s2_ndwi_threshold).clip(extent).rename("flood_mask")

    # Copy properties to preserve metadata
    mask = mask.copyProperties(image, ["system:time_start"])
    
    # Return the mask, clipped to AOI
    return mask

In [26]:
# Apply the function to extract masks from all imagery Sentinel-2 collection
s2_flood_mask_col = s2_img_col.map(lambda img: extract_floodmask_s2(img, s2_ndwi_threshold=0.1, extent= aoi_geometry))

In [27]:
# Convert Sentinel-2 flood mask collection to a List for visualization
s2_flood_mask_list = s2_flood_mask_col.toList(s2_flood_mask_col.size())
flood_mask_count = s2_flood_mask_col.size().getInfo()

print(f"There are {flood_mask_count} flood masks in the collection")

There are 15 flood masks in the collection


In [28]:
# Visualize Sentinel 2 Flood mask
s2_flood_map = geemap.Map(center=map_center_coords, zoom=11)
s2_flood_map.add_basemap("SATELLITE")

s2_flood_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s2_flood_map.addLayer(s2_composite, vis_params_s2_fcc, "S2 Scaled Masked Image")

# Loop through all flood masks and add them to the map
for i in range(flood_mask_count):
    mask_img = ee.Image(s2_flood_mask_list.get(i))
    try:
        date = ee.Date(mask_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    except:
        date = f"Mask_{i+1}"
    
    s2_flood_map.addLayer(
        mask_img, 
        vis_params_flood, 
        f"S2 Flood Mask - {date}"
    )
    
# Add composite flood mask
s2_flood_composite = s2_flood_mask_col.max()
s2_flood_map.addLayer(
    s2_flood_composite, vis_params_flood, "Sentinel-1 Maximum Flood Mask")



s2_flood_map


Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [29]:
# Flood extraction function for Sentinel-1

def extract_floodmask_s1(image: ee.Image, s1_threshold: float, polarization: str = "VV", extent: ee.Feature | ee.Geometry = None):
    """
    Extract flood mask from Sentinel-1 imagery using backscatter thresholding.
    
    Flooded areas typically show lower backscatter values (dark) compared to non-flooded areas.
    
    Args:
        image (ee.Image): Sentinel-1 image (should have VV and/or VH bands)
        s1_threshold (float): Backscatter threshold for water detection (in dB).
                               Lower values (more negative) detect more water.
                               Typical range: -25 to -10 dB.
        polarization (str): Polarization band to use ('VV' or 'VH'). Default is 'VV'.
        extent (ee.Feature|ee.Geometry): The region to clip the mask to.
        
    Returns:
        ee.Image: Binary flood mask (1 = flood, 0 = non-flood).
    """
    
    # Create binary mask where backscatter is less than threshold
    flood_mask = image.select(polarization).lt(s1_threshold).clip(extent).rename("flood_mask")
    
    
    # Copy properties to preserve metadata
    flood_mask = flood_mask.copyProperties(image, ["system:time_start"])
    
    return flood_mask

In [30]:
# Apply the function to Sentinel-1 collection
s1_flood_mask_col = s1_img_col.map(lambda img: extract_floodmask_s1(
    img, 
    s1_threshold=-15,  # Adjust threshold based on histogram analysis
    polarization="VV",
    extent=aoi_geometry
))

# Convert to list for visualization
s1_flood_mask_list = s1_flood_mask_col.toList(s1_flood_mask_col.size())
s1_flood_mask_count = s1_flood_mask_col.size().getInfo()
print(f"There are {s1_flood_mask_count} flood masks from Sentinel-1")

There are 4 flood masks from Sentinel-1


In [31]:
# Visualize Sentinel-1 Flood masks
s1_flood_map = geemap.Map(center=map_center_coords, zoom=11)
s1_flood_map.add_basemap("SATELLITE")

s1_flood_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
s1_flood_map.addLayer(s1_composite, vis_params_s1_vv, "Sentinel-1 VV (Filtered)")

# Add individual flood masks
for i in range(s1_flood_mask_count):
    flood_img = ee.Image(s1_flood_mask_list.get(i))
    try:
        date = ee.Date(flood_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    except:
        date = f"Mask_{i+1}"
    
    s1_flood_map.addLayer(
        flood_img,
        vis_params_flood,
        f"S1 Flood Mask - {date}"
    )

# Add composite flood mask
s1_flood_composite = s1_flood_mask_col.max()
s1_flood_map.addLayer(
    s1_flood_composite,
    vis_params_flood,
    "Sentinel-1 Maximum Flood Mask"
)

s1_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

*Flood extraction for Landsat-8 using Thresholding approach*

In [32]:
# Flood extraction function for Landsat-8

def extract_floodmask_l8(image: ee.Image, l8_threshold: float, extent: ee.Feature | ee.Geometry):
    """
    Extract flood mask from Landsat-8 imagery using NDWI thresholding approach.

    Args:
        image (ee.Image): Landsat-8 image
        l8_threshold (float): NDWI threshold for water detection
        extent (ee.Feature | ee.Geometry): Area of interest

    Returns:
        ee.Image: Binary flood mask (1 = flood, 0 = non-flood)
    """
    # Create binary flood mask: 1 = flood, 0 = non-flood
    flood_mask = image.select("ndwi").gt(l8_threshold).rename("flood_mask")

    # Keep BOTH classes (no masking!)
    return flood_mask.clip(extent) \
                     .copyProperties(image, ["system:time_start"])

In [33]:
# Apply flood mask extraction to all Landsat-8 images
l8_flood_mask_col = l8_img_col.map(lambda img: extract_floodmask_l8(img, l8_threshold=0.3, extent=aoi_geometry))

# Print number of flood masks
l8_flood_mask_count = l8_flood_mask_col.size().getInfo()
print(f"There are {l8_flood_mask_count} flood masks in the collection")

There are 5 flood masks in the collection


In [34]:
# Convert Landsat-8 flood mask collection to a list
l8_flood_mask_list = l8_flood_mask_col.toList(l8_flood_mask_col.size())

In [35]:
# Visualize Landsat-8 Flood mask
l8_flood_map = geemap.Map(center=map_center_coords, zoom=11)
l8_flood_map.add_basemap("SATELLITE")

l8_flood_map.addLayer(aoi_feature, vis_params_aoi, "AOI")
l8_flood_map.addLayer(l8_composite, vis_params_l8_fcc, "L8 Scaled Masked Image")


# Loop through all flood masks and add them to the map
for i in range(l8_flood_mask_count):
    flood_img = ee.Image(l8_flood_mask_list.get(i))
    try:
        date = ee.Date(flood_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    except:
        date = f"Mask_{i+1}"

    l8_flood_map.addLayer(
        flood_img,
        vis_params_flood,
        f"L8 Flood Mask {date}"
    )

# Calculate and add maximum flood mask
l8_flood_mask_max = l8_flood_mask_col.max()

l8_flood_map.addLayer(
    l8_flood_mask_max,
    vis_params_flood,
    "L8 Maximum Flood Mask"
)

l8_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Thematic Layers Downloading

*Download WorldPop Population Data*

In [36]:
# Function to download WorldPop population data 
def download_worldpop_new(year: int, extent: ee.Feature | ee.Geometry):
    """
    Download WorldPop population data for a given year and extent.

    Args:
        year (int): Year of population data (e.g. 2020, 2022)
        extent (ee.Feature | ee.Geometry): Area of interest

    Returns:
        ee.ImageCollection: Filtered WorldPop image collection
    """
    img_collection = (
        ee.ImageCollection("projects/sat-io/open-datasets/WORLDPOP/pop")
        .filter(ee.Filter.calendarRange(year, year, "year"))
        .filterBounds(extent)
    )

    img_count = img_collection.size().getInfo()
    print(f"There are {img_count} WorldPop image(s) for {year}")
    
    return img_collection

In [37]:
# Download 2020 and 2022 WorldPop data
wp_2020_col = download_worldpop_new(2020, aoi_geometry)
wp_2022_col = download_worldpop_new(2022, aoi_geometry)

# Convert collections to images
pop_2020 = wp_2020_col.mosaic().clip(aoi_geometry)
pop_2022 = wp_2022_col.mosaic().clip(aoi_geometry)

There are 3 WorldPop image(s) for 2020
There are 3 WorldPop image(s) for 2022


In [38]:
#Visualize Population density
pop_map = geemap.Map(center=map_center_coords, zoom=10)
pop_map.add_basemap("SATELLITE")

pop_map.addLayer(pop_2020, vis_params_pop, "WorldPop 2020")
pop_map.addLayer(pop_2022, vis_params_pop, "WorldPop 2022")
pop_map.addLayer(aoi_geometry, {"color": "blue"}, "AOI")

pop_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

*Download ESA World Cover Data*

In [39]:
# Function to download ESA WorldCover land cover data for 2021
def download_worldcover_2021(extent: ee.Feature | ee.Geometry):
    """
    Download ESA WorldCover 2021 land cover data for a given extent.
    
    ESA WorldCover v200 provides 10m resolution global land cover maps for 2021.
    Reference: https://esa-worldcover.org/en
    
    Args:
        extent (ee.Feature | ee.Geometry): Area of interest

    Returns:
        ee.Image: Clipped ESA WorldCover 2021 image with land cover classes
"""
    # Load ESA WorldCover 2021 collection (v200 is the 2021 product)
    worldcover_img = (
        ee.ImageCollection("ESA/WorldCover/v200")
        .first()  # Get 2021 image
        .clip(extent)
    )
    
    return worldcover_img

In [40]:
# Download ESA WorldCover 2021

landcover_2021 = download_worldcover_2021(extent=aoi_geometry)


In [41]:
# Create thematic map
thematic_map = geemap.Map(center=map_center_coords, zoom=10)
thematic_map.add_basemap("SATELLITE")

# Add layers
thematic_map.addLayer(landcover_2021, vis_params_landcover, "ESA WorldCover 2021")
thematic_map.addLayer(aoi_geometry, {"color": "pink"}, "AOI")

# Display map
thematic_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

### . Download Global Surface Water (GSW) (Permanent water)

In [42]:
def download_gsw(extent: ee.Feature | ee.Geometry, seasonality_threshold: int):
    """
    Download Global Surface Water (GSW) dataset and filter by seasonality.
    
    Args:
        extent (ee.Feature|ee.Geometry): The region to filter the collection by.
        seasonality_threshold (int): Number of months water presence (1-12). 

    Returns:
        ee.Image: The filtered GSW image with water presence layer.
    """
    # Load Global Surface Water dataset
    gsw = (ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
                  .select('seasonality')      # seasonality layer
                  .gte(seasonality_threshold) # Seasonality threshold ranges from 0-12 months
                  .clip(extent)               # Clipped to AOI
    )

    
    print(f"Permanent water loaded")
    
    return gsw


In [43]:
# Download Global Surface Water data
perm_water = download_gsw(aoi_geometry, seasonality_threshold=10)

Permanent water loaded


In [44]:
# Visualize Perm water
s2_flood_map.addLayer(perm_water, vis_params_perm_water, "Permanet Water")
s2_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

**Download SRTM DEM**

In [45]:
aoi_coords = [
    [
        [6.853409, 7.676968],
        [6.685181, 7.676968],
        [6.685181, 7.890588],
        [6.853409, 7.890588],
        [6.853409, 7.676968]
    ]
]

aoi_geom = ee.Geometry.Polygon(aoi_coords)

extent = aoi_geom


In [46]:
# Function to download SRTM DEM
def download_srtm(extent: ee.Geometry):
    """
    Download SRTM DEM for a given geographic extent.

    Args:
        extent (ee.Geometry): The region to clip the DEM to.

    Returns:
        ee.Image: The clipped SRTM DEM image.
    """
    dem = ee.Image("USGS/SRTMGL1_003").clip(extent)

    print("SRTM DEM loaded successfully.")

    return dem

In [47]:
srtm_dem = download_srtm(extent)

aoi_map.addLayer(srtm_dem, vis_params_srtm_dem, "SRTM DEM")
aoi_map

SRTM DEM loaded successfully.


Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

**Download Copernicus DEM GLO-30**

In [48]:
# Function to download Copernicus DEM GLO-30
def download_copdem(extent: ee.Geometry):
    """
    Download Copernicus DEM GLO-30 for a given geographic extent.

    Args:
        extent (ee.Geometry): The region to clip the DEM to.

    Returns:
        ee.Image: The clipped Copernicus DEM image.
    """
    dem_collection = (ee.ImageCollection("COPERNICUS/DEM/GLO30")
                      .filterBounds(extent)
                      .select("DEM"))
    
    img_count = dem_collection.size().getInfo()
    print(f"There are {img_count} images in the Copernicus DEM collection\n")
    
    dem = dem_collection.mosaic().clip(extent)
    
    return dem

In [49]:
cop_dem = download_copdem(extent)

aoi_map.addLayer(cop_dem, vis_params_cop_dem, "Copernicus DEM GLO-30")
aoi_map

There are 1 images in the Copernicus DEM collection



Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

**Download Building Dataset**

In [50]:
# Function to download Google Open Building footprints
def download_buildings(extent=  ee.Feature | ee.Geometry):
    """
    Download Google Open Building footprints for a given area of interest.
    
    Args:
        extent: geometry or feature collection
        
    Returns:
        GeoDataFrame: Building footprints as a GeoPandas GeoDataFrame
    """
    # Load and filter building data
    buildings = (ee.FeatureCollection("GOOGLE/Research/open-buildings/v3/polygons")
                 .filterBounds(extent))
    
    # Convert to GeoDataFrame
    buildings_gdf = geemap.ee_to_gdf(buildings)
    
    return buildings_gdf

In [51]:
# Download buildings for your AOI
buildings = download_buildings(extent= aoi_geometry)

## Helper Functions

*Generates and Labels Random Points for Landsat-8*

In [52]:
def generate_stratified_samples(
    image: ee.Image,
    class_band: str,
    num_points: int,
    region: ee.Geometry,
    scale: int,
    seed: int,
    geometries: bool
):
    """
    Generate stratified random sample points and extract pixel values from an image.

    Args:
        image (ee.Image): Input image containing predictor bands and class band.
        class_band (str): Name of the class/label band.
        num_points (int): Number of points per class.
        region (ee.Geometry): AOI for sampling.
        scale (int): Pixel resolution for sampling.
        seed (int): Random seed for reproducibility.
        geometries (bool): Whether to return sample point geometries.

    Returns:
        ee.FeatureCollection: Stratified sample points with extracted pixel values.
    """
    samples = image.stratifiedSample(
        numPoints=num_points,
        classBand=class_band,
        region=region,
        scale=scale,
        seed=seed,
        geometries=geometries
    )
    
    return samples

In [53]:
# Rename flood mask to label
l8_label_mask = l8_flood_mask_max.rename("label")
print(l8_label_mask.bandNames().getInfo())

# Combine Landsat-8 image and label band
l8_image_mask = l8_composite.addBands(l8_label_mask)
print("Bands in combined image and mask:", l8_image_mask.bandNames().getInfo())

['label']
Bands in combined image and mask: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'ndvi', 'ndwi', 'mndwi', 'evi', 'label']


In [54]:
# Generate stratified random sample points
l8_flood_sample_points = generate_stratified_samples(
    image=l8_image_mask,
    class_band="label",
    num_points=250,
    region=aoi_geometry,
    scale=30,
    seed=42,
    geometries=True
)

print("Sample size:", l8_flood_sample_points.size().getInfo())
print("Class distribution:", l8_flood_sample_points.aggregate_histogram("label").getInfo())

Sample size: 500
Class distribution: {'0': 250, '1': 250}


In [55]:
# Inspect the samples
print("First sample point:", l8_flood_sample_points.first().getInfo())

First sample point: {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Point', 'coordinates': [6.705968511716435, 7.830569415905663]}, 'id': '0', 'properties': {'QA_PIXEL': 21824, 'QA_RADSAT': 0, 'SR_B1': 0.01792374999999999, 'SR_B2': 0.023327499999999987, 'SR_B3': 0.048723749999999996, 'SR_B4': 0.0385625, 'SR_B5': 0.2380475, 'SR_B6': 0.15893, 'SR_B7': 0.07487625, 'SR_QA_AEROSOL': 160, 'ST_ATRAN': 6096, 'ST_B10': 305.17275182000003, 'ST_CDIST': 641.5, 'ST_DRAD': 1476, 'ST_EMIS': 9735.5, 'ST_EMSD': 137, 'ST_QA': 264, 'ST_TRAD': 9362.5, 'ST_URAD': 3185, 'evi': 0.38526543775850586, 'label': 0, 'mndwi': -0.5307258367538452, 'ndvi': 0.7211814522743225, 'ndwi': -0.6601930856704712}}


In [56]:
# Visualise the sample points
l8_sample_map = geemap.Map(center=map_center_coords, zoom=10)
l8_sample_map.add_basemap("SATELLITE")

l8_sample_style = {"color": "blue", "pointSize": 3}
l8_styled_points = l8_flood_sample_points.style(**l8_sample_style)

l8_sample_map.addLayer(l8_composite.clip(aoi_geometry), vis_params_l8_fcc, "Landsat-8 Image")
l8_sample_map.addLayer(l8_styled_points, {}, "Stratified Sample Points")

l8_sample_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

*Divides Sample Points to Train/Test for Landsat-8*

In [57]:
# Helper Functions

def split_train_test(samples: ee.FeatureCollection, train_ratio: float = 0.7, seed: int = 42):
    """
    Split sampled points into training and testing sets.

    Args:
        samples (ee.FeatureCollection): Sampled points to split.
        train_ratio (float): Proportion of samples for training set.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: (train_samples, test_samples)
    """
    split_samples = samples.randomColumn(columnName="random", seed=seed)

    train_samples = split_samples.filter(ee.Filter.lt("random", train_ratio))
    test_samples = split_samples.filter(ee.Filter.gte("random", train_ratio))

    return train_samples, test_samples

In [58]:
# Combine Landsat-8 image and weak flood mask from thresholding
l8_label_mask = l8_flood_mask_max.rename("label")
print(l8_label_mask.bandNames().getInfo())

l8_image_mask = l8_composite.addBands(l8_label_mask)
print("Bands in combined image and mask:", l8_image_mask.bandNames().getInfo())

['label']
Bands in combined image and mask: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'ndvi', 'ndwi', 'mndwi', 'evi', 'label']


In [59]:
# Create random points and extract pixel values and flood/non-flood labels
flood_sample_points_l8 = l8_image_mask.stratifiedSample(
    numPoints=250,
    classBand="label",
    region=aoi_geometry,
    scale=30,
    seed=42,
    geometries=True
)

print("Class distribution:", flood_sample_points_l8.aggregate_histogram("label").getInfo())

Class distribution: {'0': 250, '1': 250}


In [60]:
# Split into train/test
train_l8, test_l8 = split_train_test(flood_sample_points_l8, train_ratio=0.7, seed=42)

print(f"The training sample size: {train_l8.size().getInfo()}")
print(f"The testing sample size: {test_l8.size().getInfo()}")

The training sample size: 362
The testing sample size: 138


In [61]:
# Inspect them
print(f"The training sample example: {train_l8.first().getInfo()}")
print(f"The testing sample example: {test_l8.first().getInfo()}")

training_classes_l8 = train_l8.aggregate_array("label").distinct()
testing_classes_l8 = test_l8.aggregate_array("label").distinct()

print("Unique Classes in Training Set:", training_classes_l8.getInfo())
print("Unique Classes in Testing Set:", testing_classes_l8.getInfo())

The training sample example: {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Point', 'coordinates': [6.705968511716435, 7.830569415905663]}, 'id': '0', 'properties': {'QA_PIXEL': 21824, 'QA_RADSAT': 0, 'SR_B1': 0.01792374999999999, 'SR_B2': 0.023327499999999987, 'SR_B3': 0.048723749999999996, 'SR_B4': 0.0385625, 'SR_B5': 0.2380475, 'SR_B6': 0.15893, 'SR_B7': 0.07487625, 'SR_QA_AEROSOL': 160, 'ST_ATRAN': 6096, 'ST_B10': 305.17275182000003, 'ST_CDIST': 641.5, 'ST_DRAD': 1476, 'ST_EMIS': 9735.5, 'ST_EMSD': 137, 'ST_QA': 264, 'ST_TRAD': 9362.5, 'ST_URAD': 3185, 'evi': 0.38526543775850586, 'label': 0, 'mndwi': -0.5307258367538452, 'ndvi': 0.7211814522743225, 'ndwi': -0.6601930856704712, 'random': 0.053352565691913045}}
The testing sample example: {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Point', 'coordinates': [6.740194324041388, 7.78206039056321]}, 'id': '1', 'properties': {'QA_PIXEL': 21824, 'QA_RADSAT': 0, 'SR_B1': 0.05770249999999999, 'SR_B2': 0.075467500

In [62]:
# Visualize them
l8_ml_flood_map = geemap.Map(center=map_center_coords, zoom=10)
l8_ml_flood_map.add_basemap("SATELLITE")

train_style = {"color": "blue", "pointSize": 3}
test_style = {"color": "black", "pointSize": 3}

train_layer_l8 = train_l8.style(**train_style)
test_layer_l8 = test_l8.style(**test_style)

l8_ml_flood_map.addLayer(l8_composite.clip(aoi_geometry), vis_params_l8_fcc, "Landsat-8 Image")
l8_ml_flood_map.addLayer(train_layer_l8, {}, "Train Samples")
l8_ml_flood_map.addLayer(test_layer_l8, {}, "Test Samples")

l8_ml_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

## Machine Learning

*Machine Learning Classification For Landsat-8*

In [63]:
# Create display version of threshold flood mask
l8_flood_mask_thresh = l8_flood_mask_max.updateMask(l8_flood_mask_max).rename("flood_mask")

# Spectral feature bands for Landsat-8
feature_bands_l8 = ["SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]

# Initialize RF model
rf_model = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    variablesPerSplit=3,
    minLeafPopulation=5
)

# Train RF model
rf_classifier = rf_model.train(
    features=train_l8,
    classProperty='label',
    inputProperties=feature_bands_l8
)

# Classify entire Landsat-8 image into flood or no flood
l8_flood_mask_ml = (l8_image_mask.select(feature_bands_l8)
                    .classify(rf_classifier)
                    .rename("flood_mask"))

# Mask non-flood pixels
l8_flood_mask_ml = l8_flood_mask_ml.updateMask(l8_flood_mask_ml)

# Visualization parameters for flood mask
vis_params_flood = {'min': 0, 'max': 1, 'palette': ['blue']}

# Compare Thresholding & ML flood masks
ml_flood_map_l8 = geemap.Map(center=map_center_coords, zoom=10)
ml_flood_map_l8.add_basemap("SATELLITE")

ml_flood_map_l8.addLayer(l8_composite.clip(aoi_geometry), vis_params_l8_fcc, "Landsat-8 Image")
ml_flood_map_l8.addLayer(l8_flood_mask_ml, vis_params_flood, "L8 - Flood [ML]")
ml_flood_map_l8.addLayer(
    l8_flood_mask_thresh,
    {'min': 0, 'max': 1, 'bands': ['flood_mask'], 'palette': ['white', 'green']},
    "L8 - Flood [THRESHOLD]"
)

ml_flood_map_l8

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [64]:
# Classify the test dataset using trained RF model
l8_test_classified = test_l8.classify(rf_classifier)

# Confusion matrix
l8_conf_matrix = l8_test_classified.errorMatrix('label', 'classification')

# Print results
print("Confusion Matrix:", l8_conf_matrix.getInfo())
print("Overall Accuracy:", l8_conf_matrix.accuracy().getInfo())
print("Kappa:", l8_conf_matrix.kappa().getInfo())

Confusion Matrix: [[65, 8], [0, 65]]
Overall Accuracy: 0.9420289855072463
Kappa: 0.8844463052124762


**Sentinel-2**

In [65]:
# Create a median composite from the flood mask collection
s2_flood_composite = s2_flood_mask_col.max()

# Create a resampled composite
s2_composite_resampled = s2_composite  # Use existing composite

# Define the flood mask from thresholding (using maximum composite)
s2_flood_mask_thresh = s2_flood_composite.rename("flood_mask")

# Create label mask by renaming the flood mask
s2_label_mask = s2_flood_mask_thresh.rename("label")

# Combine S2 image and weak flood masks from thresholding method
# Use the composite image as the base
s2_image_mask = s2_composite_resampled.addBands(s2_label_mask)

print("Bands in combined image and mask:", s2_image_mask.bandNames().getInfo())

Bands in combined image and mask: ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'ndvi', 'ndwi', 'mndwi', 'evi', 'label']


In [66]:
# Create random points and extract band/pixel values and flood/non-flood labels
s2_flood_sample_points = generate_stratified_samples(
    image=s2_image_mask,
    num_points=250,
    class_band="label",
    region=aoi_geometry,
    scale=10,
    seed=42,
    geometries=True
)
print("Sample size:", s2_flood_sample_points.size().getInfo())
print("Class distribution:", s2_flood_sample_points.aggregate_histogram("label").getInfo())

Sample size: 500
Class distribution: {'0': 250, '1': 250}


In [67]:
# Visualise the sample points
s2_sample_map = geemap.Map(center=map_center_coords, zoom=11)
s2_sample_map.add_basemap("SATELLITE")

s2_sample_style = {"color": "blue", "pointSize": 3}
sample_layer_l8 = flood_sample_points_l8.style(**s2_sample_style)

s2_sample_map.addLayer(s2_composite.clip(aoi_geometry), vis_params_s2_fcc, "Sentinel-2 Image")
s2_sample_map.addLayer(sample_layer_l8, {}, "Stratified Sample Points")

s2_sample_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [68]:
# Split into train/test
s2_train, s2_test = split_train_test(s2_flood_sample_points, train_ratio=0.7, seed=42)

print(f"The training sample size: {s2_train.size().getInfo()}")
print(f"The testing sample size: {s2_test.size().getInfo()}")

The training sample size: 361
The testing sample size: 139


In [69]:
# Inspect train and test data
print(f"The training sample Example: {s2_train.first().getInfo()}")
print(f"The testing sample Example: {s2_test.first().getInfo()}")

# Extract unique class values
s2_training_classes = s2_train.aggregate_array("label").distinct()
s2_testing_classes = s2_test.aggregate_array("label").distinct()

print("Unique Classes in Training Set:", s2_training_classes.getInfo())
print("Unique Classes in Testing Set:", s2_testing_classes.getInfo())

The training sample Example: {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Point', 'coordinates': [6.774420136366341, 7.7202562990157855]}, 'id': '0', 'properties': {'B11': 0.29825, 'B12': 0.22015, 'B2': 0.0689, 'B3': 0.09855, 'B4': 0.1066, 'B5': 0.16115000000000002, 'B6': 0.2479, 'B7': 0.27765, 'B8': 0.28380000000000005, 'B8A': 0.3062, 'evi': 0.3213012150132561, 'label': 0, 'mndwi': -0.4991689920425415, 'ndvi': 0.45673856139183044, 'ndwi': -0.4774158000946045, 'random': 0.49508783794670763}}
The testing sample Example: {'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Point', 'coordinates': [6.8056815082537, 7.749631208806494]}, 'id': '1', 'properties': {'B11': 0.27035, 'B12': 0.19180000000000003, 'B2': 0.08310000000000001, 'B3': 0.1033, 'B4': 0.11460000000000001, 'B5': 0.14315, 'B6': 0.22190000000000001, 'B7': 0.2528, 'B8': 0.24910000000000002, 'B8A': 0.27965, 'evi': 0.25787929135882864, 'label': 0, 'mndwi': -0.4247991442680359, 'ndvi': 0.36980223655700684, 

In [70]:
# Visualize the training and testing sets
s2_ml_flood_map = geemap.Map(center=map_center_coords, zoom=10)
s2_ml_flood_map.add_basemap("SATELLITE")

train_style = {"color": "blue", "pointSize": 3}
test_style = {"color": "purple", "pointSize": 3}

s2_train_layer = s2_train.style(**train_style)
s2_test_layer = s2_test.style(**test_style)

s2_ml_flood_map.addLayer(s2_composite_resampled.clip(aoi_geometry), vis_params_s2_fcc, "S2 Image")
s2_ml_flood_map.addLayer(s2_train_layer, {}, "Train Samples")
s2_ml_flood_map.addLayer(s2_test_layer, {}, "Test Samples")

s2_ml_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [71]:
# Define spectral feature bands for Sentinel-2
s2_feature_bands = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]

# Initialize Random Forest model
rf_model = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    variablesPerSplit=3,
    minLeafPopulation=5
)

# Train Random Forest model
rf_classifier = rf_model.train(
    features=s2_train,
    classProperty='label',
    inputProperties=s2_feature_bands
)

# Classify entire S2 image into flood or no flood
s2_flood_mask_ml = (s2_image_mask.select(s2_feature_bands)
                    .classify(rf_classifier)
                    .rename("flood_mask"))

# Mask non-flood pixels
s2_flood_mask_ml = s2_flood_mask_ml.updateMask(s2_flood_mask_ml)


# Compare Thresholding & ML flood masks
s2_ml_flood_map.addLayer(s2_flood_mask_thresh, {'min': 0, 'max': 1, 'bands': ['flood_mask'], 'palette': ['white', 'green']}, "S2 - Flood [THRESHOLD]")
s2_ml_flood_map.addLayer(s2_flood_mask_ml, vis_params_flood, "S2 - Flood [ML]")

s2_ml_flood_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [72]:
# Accuracy Assessment
s2_test_classified = s2_test.classify(rf_classifier)

s2_conf_matrix = s2_test_classified.errorMatrix('label', 'classification')

print("Confusion Matrix:", s2_conf_matrix.getInfo())
print("Overall Accuracy:", s2_conf_matrix.accuracy().getInfo())
print("Kappa:", s2_conf_matrix.kappa().getInfo())

Confusion Matrix: [[66, 4], [2, 67]]
Overall Accuracy: 0.9568345323741008
Kappa: 0.9136824673980544


## . Area of Raster Layer

In [73]:
# Function to calculate the area of raster layer

def compute_raster_area(image: ee.Image, aoi: ee.Geometry, scale:int =10):
    """
    Calculate the area of a raster layer (e.g., flood mask) in square kilometers (km²).
    
    This function computes the total area of pixels matching a specified value
    within a given area of interest.
    
    Args:
        image (ee.Image): Binary or classified raster image
        aoi (ee.Geometry): Area of interest boundary
        scale (int): Pixel resolution in meters (default: 10 for Sentinel-2)
        
    Returns:
        float: Area in square kilometers (km²)
    """
    # Create an area image (pixel area in square meters m²)
    area_image = image.multiply(ee.Image.pixelArea())
    
    # Sum the area over the AOI
    total_area = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=scale,
        maxPixels=1e13
    )
    
    # Convert from m2 to km2 (divide by 1,000,000)
    area_km2 = ee.Number(total_area.get(image.bandNames().get(0))).divide(1e6)
    
    return area_km2

In [74]:
# Compute area for s2 ML-based flood mask
s2_ml_area = compute_raster_area(s2_flood_mask_ml.selfMask(), aoi_geometry)
print(f"Sentinel-2 ML Flood Area: {s2_ml_area.getInfo():.2f} km2")

# Compute area for  S2 threshold-based flood mask
s2_threshold_area = compute_raster_area(s2_flood_mask_thresh.selfMask(), aoi_geometry)
print(f"Threshold-based Flood Area (S2): {s2_threshold_area.getInfo():.2f} km²")

Sentinel-2 ML Flood Area: 134.13 km2
Threshold-based Flood Area (S2): 127.35 km²


## . Damage / Impact Assessment

In [ ]:
# Function to assess the imoact of flood on Landcover, buildings and population

def compute_flood_impact(flood_mask: ee.Image, thematic_layer: ee.Image | ee.FeatureCollection, aoi: ee.Geometry, scale: int, metric_type: str = 'area'):
    """
    Calculates the impact of flooding on various thematic layers.
    
    Args:
        flood_mask: Binary flood mask (1 = flooded).
        thematic_layer: The layer to analyze.
        aoi: Area of Interest.
        scale: Spatial resolution for analysis.
        metric_type: 'area' (for landcover/buildings) or 'population' (for sum of counts).
        
    Returns:
        float: The calculated impact value (km2 for area, total count for population).
    """
    # RASTER LOGIC (Land Cover / Population)
    if isinstance(thematic_layer, ee.Image):
        affected_layer = thematic_layer.updateMask(flood_mask)
        if metric_type == 'area':
            stats = flood_mask.multiply(ee.Image.pixelArea()).reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=scale, maxPixels=1e13)
            return ee.Number(stats.get(flood_mask.bandNames().get(0))).divide(1e6)
        elif metric_type == 'population':
            stats = affected_layer.reduceRegion(
                reducer=ee.Reducer.sum(), geometry=aoi, scale=scale, maxPixels=1e13)
            return ee.Number(stats.get(thematic_layer.bandNames().get(0)))

    # VECTOR LOGIC (Buildings)
    elif isinstance(thematic_layer, ee.FeatureCollection):
        # Create a geometry from the flood mask to filter buildings
        flood_vec = flood_mask.selfMask().reduceToVectors(geometry=aoi, scale=scale, maxPixels=1e13)
        affected_buildings = thematic_layer.filterBounds(flood_vec.geometry())
        
        if metric_type == 'area':
            # Sum the area of each individual building polygon and convert to km2
            total_area = affected_buildings.map(lambda f: f.set('area_m2', f.area()))
            return ee.Number(total_area.aggregate_sum('area_m2')).divide(1e6)
        else:
            # Default to count
            return ee.Number(affected_buildings.size())

    return ee.Number(0)

**Land cover Impact**

In [94]:
# Impact of Flood on Land Cover (Area in km2)
lc_impact_km2 = compute_flood_impact(s2_flood_mask_ml, landcover_2021, aoi_geometry, 10, 'area')

print(f"Total Land Cover Affected: {lc_impact_km2.getInfo():.2f} km²")

Total Land Cover Affected: 134.13 km²


In [96]:
# Define the ESA WorldCover v200 class mapping
lc_class_names = {
    10: "Trees",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare / sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen"
}

# Calculate area grouped by land cover class
# Divide by 1e6 for km2 and group by the land cover band
stats = ee.Image.pixelArea().divide(1e6).addBands(landcover_2021) \
    .updateMask(s2_flood_mask_ml) \
    .reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='class'),
        geometry=aoi_geometry,
        scale=10,
        maxPixels=1e13
    ).get('groups').getInfo()

# Print the results
print("Flooded Area per Land Cover Class:")
for item in stats:
    class_id = item['class']
    area_km2 = item['sum']
    name = lc_class_names.get(class_id, "Unknown")
    print(f"- {name}: {area_km2:.2f} km²")

Flooded Area per Land Cover Class:
- Trees: 28.93 km²
- Shrubland: 12.55 km²
- Grassland: 22.06 km²
- Cropland: 20.14 km²
- Built-up: 1.51 km²
- Bare / sparse vegetation: 1.04 km²
- Permanent water bodies: 43.96 km²
- Herbaceous wetland: 3.94 km²


**Building Impact**

In [97]:
# Impact of Flood on Building (Area of Buildings Affected in km2)

building_impact_km2 = compute_flood_impact(s2_flood_mask_ml, buildings, aoi_geometry, 10, 'area')

print(f"Total Building Area Affected: {building_impact_km2.getInfo():.2f} km²")


Total Building Area Affected: 0.00 km²


### . Population Impact

In [98]:
# Impact of Flood onPopulation Impact (Total People Affected)
pop_impact_count = compute_flood_impact(s2_flood_mask_ml, pop_2022, aoi_geometry, 100, 'population')

print(f"Total Population Affected: {pop_impact_count.getInfo():.0f} people")

Total Population Affected: 24191 people


In [100]:
# Create a visual layer of affected areas only
affected_lc = landcover_2021.updateMask(s2_flood_mask_ml)
affected_pop = pop_2022.updateMask(s2_flood_mask_ml)

#Visualize flooding impact
impact_map = geemap.Map(center=map_center_coords, zoom=10)
impact_map.add_basemap("SATELLITE")

impact_map.addLayer(aoi_feature, vis_params_aoi, 'AOI')
impact_map.addLayer(landcover_2021, vis_params_landcover, "Landcover")
impact_map.addLayer(affected_lc, vis_params_landcover, "Affected Landcover")
impact_map.addLayer(pop_2022,vis_params_pop, "Population Density" )
impact_map.addLayer(affected_pop, {'min': 0, 'max': 10, 'palette': ['white', 'red']}, "Affected Population Density")

impact_map

Map(center=[7.8175, 6.773], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

# . Export Features

In [103]:
# Function to export Image to Drive
def export_image_to_drive(image: ee.Image, description: str, folder: str, extent: ee.Geometry, scale: int, crs: str = 'EPSG:4326'):
    """
    Starts a Google Earth Engine batch task to export a raster image to Google Drive.

    Args:
        image (ee.Image): The Earth Engine image to be exported.
        description (str): A human-readable name for the task and the output file prefix.
        folder (str): The name of the Google Drive folder where the file will be saved.
        extent (ee.Geometry): The regional boundary (AOI) to clip the export to.
        scale (int): The resolution in meters per pixel (e.g., 10 for Sentinel-2).
        crs (str): The coordinate reference system.

    Returns:
        None: The function initiates an asynchronous background task.
    """
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        region=extent,
        scale=scale,
        crs=crs,
        maxPixels=1e13
    )
    task.start()
    print(f"Started export: {description}")

In [104]:
# Function to export FeatureCollection to Drive
def export_vector_to_drive(feature_col: ee.FeatureCollection, description: str, folder: str, file_format: str, crs: str = 'EPSG:4326'):
    """
    Starts a Google Earth Engine batch task to export a FeatureCollection to Google Drive.

    Args:
        feature_col (ee.FeatureCollection): The vector data to be exported.
        description (str): A human-readable name for the task and the output file prefix.
        folder (str): The name of the Google Drive folder where the file will be saved.
        file_format (str, optional): The output format (e.g., 'SHP', 'CSV', 'KML', 'GeoJSON'). 
        crs (str): The coordinate reference system.

    Returns:
        None: The function initiates an asynchronous background task.
    """
    task = ee.batch.Export.table.toDrive(
        collection=feature_col,
        description=description,
        folder=folder,
        fileNamePrefix=description,
        fileFormat=file_format,
        crs=crs
    )
    task.start()
    print(f"Started vector export: {description}")

In [ ]:
# Define UTM CRS
utm_crs = 'EPSG:32631'

# Export the AOI (Vector) ---
export_vector_to_drive(
    feature_col=ee.FeatureCollection([aoi_feature]), 
    description="AOI_Boundary", 
    folder="Flood_Project_Exports", 
    file_format="SHP",
    crs=utm_crs
)

# 2. Export Satellite Composites (Raster) ---
# Export Sentinel-1 Max (Radar)
export_image_to_drive(
    image=s1_composite, 
    description="Sentinel1_Max_Composite", 
    folder="Flood_Project_Exports", 
    extent=aoi_geometry, 
    scale=10,
    crs=utm_crs
)

# Export Sentinel-2 Max (Optical)
export_image_to_drive(
    image=s2_composite, 
    description="Sentinel2_Max_Composite", 
    folder="Flood_Project_Exports", 
    extent=aoi_geometry, 
    scale=10,
    crs=utm_crs
)

# Export Landsat 8 (Optical)
export_image_to_drive(
    image=l8_composite, 
    description="Landsat8_Composite", 
    folder="Flood_Project_Exports", 
    extent=aoi_geometry, 
    scale=30,
    crs=utm_crs
)

# Export ML Flood Masks (Thematic)
# Sentinel-2 ML Mask
export_image_to_drive(
    image=s2_flood_mask_ml, 
    description="Flood_Mask_S2_ML", 
    folder="Flood_Project_Exports", 
    extent=aoi_geometry, 
    scale=10,
    crs=utm_crs
)

# Landsat 8 ML Mask
export_image_to_drive(
    image=l8_flood_mask_ml, 
    description="Flood_Mask_L8_ML", 
    folder="Flood_Project_Exports", 
    extent=aoi_geometry, 
    scale=30,
    crs=utm_crs
)